## DoS-Classifier

In [1]:
import pandas as pd
import numpy as np

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

PATH = 'SUEE1.csv'
PATH = '/home/tdokkad/Downloads/' + PATH

print("Loading SUEE1 dataset...")
df = pd.read_csv(PATH)
print(df.head())

slowloris = np.asarray(["10.128.0." + str(i) for i in range(1, 50)])
slowhttptest = np.asarray(["10.128.0." + str(i) for i in range(51, 100)])
slowloris_ng = np.asarray(["10.128.0." + str(i) for i in range(101, 150)])

# if you need separate index arrays for each group:
slowloris_i = np.unique(np.hstack([
    np.where(df["Source"].isin(slowloris)), 
    np.where(df["Destination"].isin(slowloris))]))
slowhttptest_i = np.unique(np.hstack([
    np.where(df["Source"].isin(slowhttptest)), 
    np.where(df["Destination"].isin(slowhttptest))]))
slowloris_ng_i = np.unique(np.hstack([
    np.where(df["Source"].isin(slowloris_ng)), 
    np.where(df["Destination"].isin(slowloris_ng))]))

# for ip in df["Source"].values:
#     if ip in slowloris:
#         df["Label"] = "slowloris"
#     elif ip in slowhttptest:
#         df["Label"] = "slowhttptest"
#     elif ip in slowloris_ng:
#         df["Label"] = "slowloris_ng"
#     else:
#         df["Label"] = "normal"

# for ip in df["Destination"].values:
#     if ip in slowloris:
#         df["Label"] = "slowloris"
#     elif ip in slowhttptest:
#         df["Label"] = "slowhttptest"
#     elif ip in slowloris_ng:
#         df["Label"] = "slowloris_ng"
#     else:
#         df["Label"] = "normal"

# Split samples into training and test sets
# feat_train, feat_test, y_train, y_test = train_test_split(
#     feat_all, y_all, test_size=0.4, random_state=0
# )


print(slowloris_i.shape)
print(slowhttptest_i.shape)
print(slowloris_ng_i.shape)
print("Completed.")

Loading SUEE1 dataset...
   No.      Time       Source  Destination Protocol  Length  \
0    1  0.000000  192.168.0.1  192.168.0.2      TCP      66   
1    2  0.038679  192.168.0.2  192.168.0.1      TCP      66   
2    3  0.156902  192.168.0.1  192.168.0.3      TCP      54   
3    4  0.158266  192.168.0.3  192.168.0.1      TCP      54   
4    5  0.593252  192.168.0.1  192.168.0.4      TCP      66   

                                                Info  
0  80  >  39266 [FIN, ACK] Seq=1 Ack=1 Win=235 Le...  
1  39266  >  80 [ACK] Seq=1 Ack=2 Win=245 Len=0 T...  
2   80  >  9784 [FIN, ACK] Seq=1 Ack=1 Win=237 Len=0  
3        9784  >  80 [ACK] Seq=1 Ack=2 Win=254 Len=0  
4  80  >  62170 [FIN, ACK] Seq=1 Ack=1 Win=235 Le...  
(2204,)
(6586,)
(17702,)
Completed.


In [3]:
X = df.copy()

HOST_ADDRESS = '192.168.0.1'
ATTACKER_PREFIX = '10.128'

# Features to mark whether traffic is outbound or inbound. 
X['Inbound'] = (X['Destination'] == HOST_ADDRESS)
X['Outbound'] = (X['Source'] == HOST_ADDRESS)

# Extract labels -- 'True' if malicious (prefixed with 10... instead of 192...)
y = np.logical_or(
    X['Source'].to_numpy().astype('U2') == '10',
    X['Destination'].to_numpy().astype('U2') == '10')

# TODO: Record an 'id' column for different clients, randomized between benign and attacker traffic? or remove entirely

X.drop(['Source', 'Destination'], axis=1, inplace=True)

Xy = X.copy()
Xy.insert(0, 'Label', y)
display(Xy)

,Label,No.,Time,Protocol,Length,Info,Inbound,Outbound
0,False,1,0.000000,TCP,66,"80 > 39266 [FIN, ACK] Seq=1 Ack=1 Win=235 Le...",False,True
1,False,2,0.038679,TCP,66,39266 > 80 [ACK] Seq=1 Ack=2 Win=245 Len=0 T...,True,False
2,False,3,0.156902,TCP,54,"80 > 9784 [FIN, ACK] Seq=1 Ack=1 Win=237 Len=0",False,True
3,False,4,0.158266,TCP,54,9784 > 80 [ACK] Seq=1 Ack=2 Win=254 Len=0,True,False
4,False,5,0.593252,TCP,66,"80 > 62170 [FIN, ACK] Seq=1 Ack=1 Win=235 Le...",False,True
...,...,...,...,...,...,...,...,...
2089431,False,2089432,464.968751,TCP,66,[TCP ACKed unseen segment] [TCP Previous segme...,True,False
2089432,False,2089433,465.155756,TCP,54,[TCP ACKed unseen segment] [TCP Previous segme...,True,False
2089433,False,2089434,465.888553,TCP,66,"80 > 42640 [PSH, ACK] Seq=1 Ack=468 Win=3008...",False,True
2089434,False,2089435,466.022984,TCP,74,[TCP Port numbers reused] 48110 > 80 [SYN] S...,True,False


In [ ]:
# Preview a random-ish set of 'info' content
for rand_index in np.random.randint(0, len(X), 20):
    print(X['Info'][rand_index])

print(X)

[TCP Dup ACK 759940#1] 23970  >  443 [PSH, ACK] Seq=599 Ack=4926 Win=43776 Len=0 TSval=1219671382 TSecr=482840251
[TCP ACKed unseen segment] [TCP Previous segment not captured] 46580  >  80 [ACK] Seq=364 Ack=1410 Win=32128 Len=0 TSval=478577295 TSecr=477337202
[TCP Dup ACK 2062148#1] 80  >  50977 [PSH, ACK] Seq=1 Ack=379 Win=30016 Len=0
[TCP ACKed unseen segment] 44008  >  80 [ACK] Seq=178 Ack=44805 Win=119040 Len=0 TSval=479164287 TSecr=718299060
[TCP ACKed unseen segment] 80  >  39260 [ACK] Seq=1 Ack=364 Win=30080 Len=0 TSval=492132713 TSecr=493372806
[TCP ACKed unseen segment] 39894  >  80 [ACK] Seq=178 Ack=15785 Win=61056 Len=0 TSval=484682221 TSecr=720506233
[TCP ACKed unseen segment] 50656  >  80 [FIN, ACK] Seq=178 Ack=52923 Win=136448 Len=0 TSval=480524400 TSecr=718843104
[TCP Previous segment not captured] 41226  >  443 [FIN, ACK] Seq=1178 Ack=4915 Win=39040 Len=0 TSval=498046197 TSecr=2057155386
[TCP Dup ACK 1206127#1] 37364  >  80 [PSH, ACK] Seq=347 Ack=1418 Win=32128 Len=0 T

In [39]:
# Break down 'info' field into structured format

# Extract bracketed arguments from start of string
def extract_front_bracketed(str):
    pre_args = []
    while str[0] == '[':
        pre_arg, str = str[1:].split(']', 1)
        pre_args.append(pre_arg)
        str = str.lstrip()
    
    return pre_args, str

# Extract bracketed arguments from end of string
def extract_hind_bracketed(str):
    post_args = []
    while str[-1] == ']':
        str, post_arg = str[:-1].split('[', 1)
        post_args.append(post_arg)
        str = str.rstrip()
    
    return post_args[::-1], str

# Process 'Info' field of converted PCAP CSV
def process_info(info):
    copy = info
    pre_args, info = extract_front_bracketed(info)
    
    # Source and destination port numbers (TODO: make sure these are being interpreted correctly)
    if len(info.split('  >  ', 1)) < 2:
        print(copy)
    source_port, info = info.split('  >  ', 1)
    dest_port, info = info.split(' ', 1)
    source_port = int(source_port)
    dest_port = int(dest_port)

    # Middle bracketed arguments
    mid_args, info = extract_front_bracketed(info)
    if len(mid_args) > 0:
        assert(len(mid_args) == 1)
        mid_args = mid_args[0].split(', ')

    # End bracketed arguments
    post_args, info = extract_hind_bracketed(info)

    vars = {}
    for var in info.split(' '):
        if '=' in var:
            k, v = var.split('=')
        else:
            k, v = var, 0
        vars[k] = int(v)

    return (pre_args, source_port, dest_port, mid_args, vars, post_args)

# Preview a random-ish set of 'info' content
a = [process_info(info) for info in X['Info']]

print(a[-10:])

# print(process_info(1))

GET /bcrypt.php HTTP/1.1 


ValueError: not enough values to unpack (expected 2, got 1)

^ this currently chokes on this line which appears to be from a different protocol. We may need to apply different processsing depending on which protocol is being used (which may significantly change the feature composition)